# 01 — EDA & Feature Engineering — Trace the Ace

Notebook A of the 4-notebook pipeline (see `Context.md`). Produces the
`trace-the-ace-features` Kaggle Dataset consumed by notebooks B, C, D:

- `session_features.csv` — one row per `session_id`, all `feat_*` columns
- `train_features_enriched.csv` — one row per `response_id`
- `cv_folds.csv` — canonical 5-fold GroupKFold split by `session_id`

No modeling happens in this notebook — EDA and features only.


## 1. Setup

In [ ]:
# Install packages not preinstalled in the Kaggle image.
# sentence-transformers is needed for the embedding-similarity feature in section 3.
# --no-deps is important: a plain `pip install sentence-transformers` pulls in a
# fresh torch build via dependency resolution that can mismatch the CUDA driver
# already baked into the Kaggle GPU image (-> "no kernel image available for
# execution on the device"). Kaggle's base image already ships torch + the rest
# of the dependency tree, so we only need the sentence-transformers package itself.
!pip install -q --no-deps sentence-transformers


In [ ]:
import os
import re
import json
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

def _resolve_dataset_dir(slug: str) -> Path:
    """Kaggle mounts a dataset at /kaggle/input/<slug>/ normally, but some
    environments nest it under /kaggle/input/datasets/<owner>/<slug>/ instead
    -- auto-detect whichever is present so this notebook (and B/C/D, which
    reuse this pattern) don't break if the mount layout differs."""
    direct = Path("/kaggle/input") / slug
    if (direct / "train_features.csv").exists():
        return direct
    nested_root = Path("/kaggle/input/datasets")
    if nested_root.exists():
        for owner_dir in nested_root.iterdir():
            candidate = owner_dir / slug
            if (candidate / "train_features.csv").exists():
                return candidate
    raise FileNotFoundError(
        f"Could not locate '{slug}' dataset under /kaggle/input "
        f"(checked {direct} and /kaggle/input/datasets/*/{slug})"
    )

RAW_DIR = _resolve_dataset_dir("trace-the-ace-raw")
TRANSCRIPTS_DIR = RAW_DIR / "train_transcripts"
print(f"Resolved RAW_DIR: {RAW_DIR}")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 5

pd.set_option("display.max_colwidth", 200)


In [ ]:
# --- Load raw data ---------------------------------------------------------
train_features = pd.read_csv(RAW_DIR / "train_features.csv")
train_labels = pd.read_csv(RAW_DIR / "train_labels.csv")

print("train_features:", train_features.shape)
print("train_labels:  ", train_labels.shape)
train_features.head()


In [ ]:
# --- Merge features + labels on response_id ---------------------------------
df = train_features.merge(train_labels, on="response_id", how="inner", validate="one_to_one")
assert len(df) == len(train_features) == len(train_labels), "merge dropped/duplicated rows"

n_responses = df["response_id"].nunique()
n_sessions = df["session_id"].nunique()
n_objectives = df["learning_objective_id"].nunique()
base_rate = df["is_correct"].mean()

print(f"responses:            {n_responses}")
print(f"sessions:             {n_sessions}")
print(f"distinct objectives:  {n_objectives}")
print(f"base rate is_correct: {base_rate:.4f}")

# Sanity check against the numbers quoted in Context.md
# (~35,072 responses / ~22,821 sessions / 398 objectives / base rate ~0.70)


## 2. Exploratory Data Analysis

In [ ]:
# --- Enumerate transcript files and load helper -----------------------------
transcript_files = sorted(TRANSCRIPTS_DIR.glob("*.csv"))
print(f"transcript files found: {len(transcript_files)}")
assert len(transcript_files) == n_sessions, "transcript file count does not match session count"

def load_transcript(session_id: str) -> pd.DataFrame:
    """Load one session's transcript CSV."""
    path = TRANSCRIPTS_DIR / f"{session_id}.csv"
    t = pd.read_csv(path)
    t = t.sort_values("utterance_id").reset_index(drop=True)
    return t

session_ids = df["session_id"].unique().tolist()


In [ ]:
# --- Load every transcript once, in parallel (IO-bound -> threads) ----------
def _load(sid):
    return sid, load_transcript(sid)

transcripts = {}
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = [ex.submit(_load, sid) for sid in session_ids]
    for fut in as_completed(futures):
        sid, t = fut.result()
        transcripts[sid] = t

print(f"loaded {len(transcripts)} transcripts into memory")

# Roles present across the corpus -- note there is a 'background' role in
# addition to 'tutor'/'student' (e.g. '[unclear]' audio artifacts). We keep it
# out of word-count / turn-count features below since it carries no student or
# tutor signal.
all_roles = pd.concat([t["role"] for t in transcripts.values()]).value_counts()
print(all_roles)


In [ ]:
# --- Transcript length (turns/session) and student word count/session -------
turn_counts = pd.Series({sid: len(t) for sid, t in transcripts.items()}, name="n_turns")

def student_word_count(t: pd.DataFrame) -> int:
    student_text = t.loc[t["role"] == "student", "content"].dropna()
    return int(student_text.str.split().str.len().sum())

student_words = pd.Series(
    {sid: student_word_count(t) for sid, t in transcripts.items()}, name="n_student_words"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(turn_counts, bins=50)
axes[0].set_title("Turns per session")
axes[0].set_xlabel("n_turns")
axes[0].set_ylabel("sessions")

axes[1].hist(student_words[student_words > 0], bins=50, log=True)
axes[1].set_title("Student word count per session (log-scale y)")
axes[1].set_xlabel("n_student_words")
axes[1].set_ylabel("sessions (log)")
plt.tight_layout()
plt.show()

print(turn_counts.describe())
print(student_words.describe())


In [ ]:
# --- is_correct distribution overall and by learning_objective --------------
print("Overall is_correct distribution:")
print(df["is_correct"].value_counts(normalize=True))

top20_objectives = df["learning_objective_id"].value_counts().head(20).index
obj_rates = (
    df[df["learning_objective_id"].isin(top20_objectives)]
    .groupby("learning_objective_id")["is_correct"]
    .agg(["mean", "count"])
    .sort_values("mean")
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(obj_rates.index.astype(str), obj_rates["mean"])
ax.axvline(base_rate, color="red", linestyle="--", label=f"overall base rate ({base_rate:.2f})")
ax.set_xlabel("mean is_correct")
ax.set_ylabel("learning_objective_id (top 20 by frequency)")
ax.set_title("Correctness rate varies substantially by objective")
ax.legend()
plt.tight_layout()
plt.show()

print(f"objective-level base rate range: {obj_rates['mean'].min():.3f} - {obj_rates['mean'].max():.3f}")
print(
    "Objective identity is a known confound (see Context.md sec 8) -- any model that "
    "conditions on transcript features alone risks partly re-deriving objective difficulty "
    "rather than session-specific signal. Worth including learning_objective_id as a "
    "feature/grouping variable downstream, or checking feature importance is not just "
    "proxying for objective."
)


In [ ]:
# --- Example transcript excerpts --------------------------------------------
# Pick one long, explanation-heavy session and one short, answer-only session.
lengths_desc = turn_counts.sort_values(ascending=False)
long_session = lengths_desc.index[0]
short_session = turn_counts[turn_counts > 2].sort_values().index[0]
mid_session = turn_counts.sort_values().index[len(turn_counts) // 2]

for label, sid in [("LONG / explanation-heavy", long_session),
                    ("SHORT / answer-only", short_session),
                    ("MEDIAN-length", mid_session)]:
    print(f"\n{'=' * 80}\n{label} SESSION: {sid}  ({turn_counts[sid]} turns)\n{'=' * 80}")
    t = transcripts[sid]
    for _, row in t.head(12).iterrows():
        print(f"[{row['role']:>10}] {row['content']}")
    if len(t) > 12:
        print(f"... ({len(t) - 12} more turns)")


In [ ]:
# --- Near-zero student-word sessions (MIN_STUDENT_WORDS open decision) ------
near_zero_threshold = 3  # candidate cutoff to evaluate; not applied/filtered here
near_zero_sessions = student_words[student_words <= near_zero_threshold].index

near_zero_responses = df[df["session_id"].isin(near_zero_sessions)]
print(f"sessions with <= {near_zero_threshold} student words: {len(near_zero_sessions)} "
      f"({len(near_zero_sessions) / n_sessions:.2%} of sessions)")
print(f"responses touched:  {len(near_zero_responses)}")
print(f"their is_correct rate: {near_zero_responses['is_correct'].mean():.4f} "
      f"(vs overall {base_rate:.4f})")
print(
    "\nFinding surfaced for later decision (Context.md sec 8: MIN_STUDENT_WORDS filtering) -- "
    "NOT dropped here. If this subgroup's correctness rate differs a lot from the base rate, "
    "notebooks B/C/D may want a dedicated fallback prediction for low-signal sessions."
)


## 3. Feature Engineering

Every feature below is implemented as a standalone function operating on one
session's transcript DataFrame (`role`, `content`, `timestamp` columns), then
applied across all sessions in parallel. All output columns are prefixed
`feat_` per the shared naming convention in `Context.md`.


In [ ]:
# --- Helpers ------------------------------------------------------------
def _words(text: str) -> list:
    if not isinstance(text, str) or not text.strip():
        return []
    return text.split()

def _role_text(t: pd.DataFrame, role: str) -> pd.Series:
    return t.loc[t["role"] == role, "content"].dropna()

def _timestamp_to_seconds(ts: str) -> float:
    h, m, s = ts.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


In [ ]:
# --- Spelled-out number word list (regex-based, no external NLP lib) --------
_ONES = ["one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
_TEENS = ["ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen",
          "seventeen", "eighteen", "nineteen"]
_TENS = ["twenty", "thirty", "forty", "fifty", "sixty", "seventy", "eighty", "ninety"]
_SCALES = ["hundred", "thousand"]

_NUMBER_WORDS = _ONES + _TEENS + _TENS + _SCALES
_NUMBER_WORD_RE = re.compile(
    r"\b(" + "|".join(_NUMBER_WORDS) + r")\b", flags=re.IGNORECASE
)

def count_number_words(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(_NUMBER_WORD_RE.findall(text))


In [ ]:
# --- Hint / correction keyword patterns (tutor turns) ------------------------
_HINT_RE = re.compile(
    r"\b(?:hint|try|think about|remember|recall|what if|consider|clue|"
    r"let'?s see|how about|what do you think|can you)\b",
    flags=re.IGNORECASE,
)
_CORRECTION_RE = re.compile(
    r"\b(?:actually|not quite|incorrect|that'?s not (?:quite )?right|let'?s try again|"
    r"mistake|wrong|close but|almost|not exactly|try again)\b",
    flags=re.IGNORECASE,
)
_QUESTION_RE = re.compile(r"\?")


In [ ]:
# --- Individual feature functions --------------------------------------------
# Each function takes one session's transcript DataFrame and returns a scalar.

def feat_n_student_words(t):
    return int(_role_text(t, "student").str.split().str.len().sum())

def feat_n_tutor_words(t):
    return int(_role_text(t, "tutor").str.split().str.len().sum())

def feat_student_word_share(t):
    n_s = feat_n_student_words(t)
    n_tu = feat_n_tutor_words(t)
    total = n_s + n_tu
    return n_s / total if total > 0 else 0.0

def feat_numeric_turns_per_word(t):
    # Fraction of (tutor+student) turns containing at least one digit,
    # normalized by total word count -- a density-of-numeric-turns signal.
    turns = t[t["role"].isin(["tutor", "student"])]
    if len(turns) == 0:
        return 0.0
    total_words = sum(len(_words(c)) for c in turns["content"])
    if total_words == 0:
        return 0.0
    numeric_turns = turns["content"].fillna("").str.contains(r"\d").sum()
    return numeric_turns / total_words

def feat_digit_chars_per_word(t):
    turns = t[t["role"].isin(["tutor", "student"])]
    if len(turns) == 0:
        return 0.0
    total_words = sum(len(_words(c)) for c in turns["content"])
    if total_words == 0:
        return 0.0
    digit_chars = turns["content"].fillna("").str.count(r"\d").sum()
    return digit_chars / total_words

def feat_number_word_count(t):
    # Spelled-out numbers in student turns (student's own numeric reasoning).
    student_text = _role_text(t, "student")
    return int(student_text.map(count_number_words).sum())

def feat_hint_count(t):
    tutor_text = _role_text(t, "tutor")
    return int(tutor_text.str.contains(_HINT_RE).sum())

def feat_correction_count(t):
    tutor_text = _role_text(t, "tutor")
    return int(tutor_text.str.contains(_CORRECTION_RE).sum())

def feat_question_count_tutor(t):
    tutor_text = _role_text(t, "tutor")
    return int(tutor_text.str.contains(_QUESTION_RE).sum())

def feat_question_count_student(t):
    student_text = _role_text(t, "student")
    return int(student_text.str.contains(_QUESTION_RE).sum())

def feat_turn_count(t):
    return int((t["role"].isin(["tutor", "student"])).sum())

def feat_avg_turn_length_student(t):
    student_text = _role_text(t, "student")
    if len(student_text) == 0:
        return 0.0
    return float(student_text.str.split().str.len().mean())

def feat_avg_turn_length_tutor(t):
    tutor_text = _role_text(t, "tutor")
    if len(tutor_text) == 0:
        return 0.0
    return float(tutor_text.str.split().str.len().mean())

def feat_turn_gap_mean_s(t):
    secs = t["timestamp"].map(_timestamp_to_seconds).sort_values()
    gaps = secs.diff().dropna()
    return float(gaps.mean()) if len(gaps) else 0.0

def feat_turn_gap_std_s(t):
    secs = t["timestamp"].map(_timestamp_to_seconds).sort_values()
    gaps = secs.diff().dropna()
    return float(gaps.std()) if len(gaps) > 1 else 0.0

def feat_late_session_student_share(t):
    # Student word share within the final third of the session (by turn order)
    # -- proxy for whether the student "takes over" reasoning near the end.
    turns = t[t["role"].isin(["tutor", "student"])].reset_index(drop=True)
    n = len(turns)
    if n == 0:
        return 0.0
    cutoff = max(int(np.ceil(n * 2 / 3)), 0)
    late = turns.iloc[cutoff:]
    n_s = int(_role_text(late, "student").str.split().str.len().sum())
    n_tu = int(_role_text(late, "tutor").str.split().str.len().sum())
    total = n_s + n_tu
    return n_s / total if total > 0 else 0.0


FEATURE_FUNCS = {
    "feat_n_student_words": feat_n_student_words,
    "feat_n_tutor_words": feat_n_tutor_words,
    "feat_student_word_share": feat_student_word_share,
    "feat_numeric_turns_per_word": feat_numeric_turns_per_word,
    "feat_digit_chars_per_word": feat_digit_chars_per_word,
    "feat_number_word_count": feat_number_word_count,
    "feat_hint_count": feat_hint_count,
    "feat_correction_count": feat_correction_count,
    "feat_question_count_tutor": feat_question_count_tutor,
    "feat_question_count_student": feat_question_count_student,
    "feat_turn_count": feat_turn_count,
    "feat_avg_turn_length_student": feat_avg_turn_length_student,
    "feat_avg_turn_length_tutor": feat_avg_turn_length_tutor,
    "feat_turn_gap_mean_s": feat_turn_gap_mean_s,
    "feat_turn_gap_std_s": feat_turn_gap_std_s,
    "feat_late_session_student_share": feat_late_session_student_share,
}


In [ ]:
# --- Apply all non-embedding features across sessions in parallel ------------
def compute_session_features(sid: str) -> dict:
    t = transcripts[sid]
    row = {"session_id": sid}
    for name, fn in FEATURE_FUNCS.items():
        row[name] = fn(t)
    return row

results = []
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = [ex.submit(compute_session_features, sid) for sid in session_ids]
    for fut in as_completed(futures):
        results.append(fut.result())

session_features = pd.DataFrame(results).set_index("session_id").loc[session_ids].reset_index()
print(session_features.shape)
session_features.head()


In [ ]:
# --- Sentence-embedding similarity feature -----------------------------------
# Uses sentence-transformers/all-MiniLM-L6-v2, which is pre-loaded in the
# competition's no-internet runtime (Context.md sec 3) -- picked specifically
# so this feature stays portable to the final submission.
#
# A session can be associated with more than one learning_objective_id
# (~37% of sessions have 2+). To keep this a genuine per-session feature we
# average the embeddings of all distinct objectives attached to a session,
# then compute mean cosine similarity of each student turn to that averaged
# objective embedding.
import torch
from sentence_transformers import SentenceTransformer

# Some Kaggle GPU sessions ship a torch build whose compiled CUDA kernels don't
# cover the accelerator actually attached to this run, which surfaces as
# "AcceleratorError: no kernel image is available for execution on the device"
# the first time a model actually runs on the GPU (model *loading* succeeds
# either way, so we can't detect this until we try a real forward pass).
# MiniLM-L6-v2 is small enough that CPU is a perfectly viable fallback, so we
# do a tiny health-check encode and downgrade to CPU if the GPU path fails,
# rather than letting the whole notebook fail on flaky accelerator infra.
_device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=_device)
try:
    embed_model.encode(["health check"], show_progress_bar=False)
    print(f"Embedding model healthy on device={_device}")
except Exception as e:
    print(f"GPU inference failed on device={_device} ({e!r}); falling back to CPU")
    _device = "cpu"
    embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=_device)
    embed_model.encode(["health check"], show_progress_bar=False)
    print("Embedding model healthy on device=cpu")

# 1. Embed every distinct learning_objective text once.
objective_lookup = (
    df[["learning_objective_id", "learning_objective"]]
    .drop_duplicates()
    .set_index("learning_objective_id")["learning_objective"]
)
objective_ids = objective_lookup.index.tolist()
objective_embeddings = embed_model.encode(
    objective_lookup.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True
)
objective_emb_map = dict(zip(objective_ids, objective_embeddings))

# 2. Per session, average the embeddings of its distinct objectives.
session_to_objectives = df.groupby("session_id")["learning_objective_id"].unique()
session_objective_emb = {
    sid: np.mean([objective_emb_map[oid] for oid in oids], axis=0)
    for sid, oids in session_to_objectives.items()
}


In [ ]:
# 3. Embed all student turns, tagged by session_id.
student_rows = []
for sid, t in transcripts.items():
    student_text = t.loc[t["role"] == "student", "content"].dropna()
    for content in student_text:
        if content.strip():
            student_rows.append((sid, content))

student_df = pd.DataFrame(student_rows, columns=["session_id", "content"])
print(f"total student turns to embed: {len(student_df)}")

# Transcripts are full of short, exactly-repeated phrases ("Yeah.", "[unclear]",
# "Okay.") -- dedup before encoding so we only pay embedding cost once per
# distinct string, then map results back to every occurrence.
unique_contents = student_df["content"].unique().tolist()
print(f"unique student turn strings: {len(unique_contents)} "
      f"({len(unique_contents) / len(student_df):.1%} of all turns)")

encode_kwargs = dict(batch_size=256, show_progress_bar=True, normalize_embeddings=True)
if _device == "cpu":
    # Multi-process CPU pool -- a single-process CPU encode of ~millions of
    # short strings is far slower than the GPU path this was designed for, so
    # spread it across all available cores when GPU inference isn't usable.
    n_workers = max(1, (os.cpu_count() or 1))
    pool = embed_model.start_multi_process_pool(target_devices=["cpu"] * n_workers)
    print(f"encoding on {n_workers} CPU worker processes")
    unique_embeddings = embed_model.encode_multi_process(
        unique_contents, pool, batch_size=256, normalize_embeddings=True
    )
    embed_model.stop_multi_process_pool(pool)
else:
    unique_embeddings = embed_model.encode(unique_contents, **encode_kwargs)

content_to_emb = dict(zip(unique_contents, unique_embeddings))
student_embeddings = np.stack([content_to_emb[c] for c in student_df["content"]])


In [ ]:
# 4. Cosine similarity (embeddings are normalized, so dot product = cosine sim)
# per turn against that turn's session-level objective embedding, then average per session.
obj_emb_matrix = np.stack(
    [session_objective_emb[sid] for sid in student_df["session_id"]]
)
sims = np.einsum("ij,ij->i", student_embeddings, obj_emb_matrix)
student_df["sim"] = sims

sim_per_session = student_df.groupby("session_id")["sim"].mean()
sim_per_session.name = "feat_embed_sim_objective_mean"

# Sessions with zero student turns get no similarity signal -- fill with the
# corpus mean so downstream models see a neutral value rather than NaN.
session_features = session_features.merge(
    sim_per_session, left_on="session_id", right_index=True, how="left"
)
session_features["feat_embed_sim_objective_mean"] = session_features[
    "feat_embed_sim_objective_mean"
].fillna(sim_per_session.mean())

FEATURE_COLS = [c for c in session_features.columns if c.startswith("feat_")]
print(f"{len(FEATURE_COLS)} feat_ columns: {FEATURE_COLS}")
session_features.head()


In [ ]:
# --- Merge engineered features onto the response-level dataframe ------------
# Session-level features repeat across responses that share a session_id --
# expected, per Context.md sec 5.
train_features_enriched = df.merge(session_features, on="session_id", how="left")
assert train_features_enriched["feat_turn_count"].isna().sum() == 0, "unmatched sessions after merge"
print(train_features_enriched.shape)
train_features_enriched.head()


In [ ]:
# --- Quintile-bucketed correctness-rate chart for every new feature ----------
def plot_quintile_correctness(frame: pd.DataFrame, feature: str, ax):
    try:
        buckets = pd.qcut(frame[feature], 5, duplicates="drop")
    except ValueError:
        # Not enough distinct values to form 5 quintiles.
        buckets = pd.cut(frame[feature], 5)
    rates = frame.groupby(buckets, observed=True)["is_correct"].mean()
    ax.bar(range(len(rates)), rates.values)
    ax.axhline(base_rate, color="red", linestyle="--", linewidth=1)
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels([f"Q{i+1}" for i in range(len(rates))], rotation=0)
    ax.set_title(feature, fontsize=9)
    ax.set_ylim(0, 1)

n_feats = len(FEATURE_COLS)
n_cols = 4
n_rows = int(np.ceil(n_feats / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(FEATURE_COLS):
    plot_quintile_correctness(train_features_enriched, feature, axes[i])
for j in range(n_feats, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


## 4. Canonical CV Fold Definition

Per `Context.md` sec 5: 5-fold `GroupKFold` grouped by `session_id`,
`random_state=42`. `GroupKFold` itself has no shuffling/`random_state`
parameter, so reproducibility is achieved by shuffling the group order with a
fixed seed before splitting -- every other notebook must load `cv_folds.csv`
and merge on `session_id` rather than re-splitting.


In [ ]:
from sklearn.model_selection import GroupKFold

rng = np.random.RandomState(RANDOM_STATE)
unique_sessions = np.array(session_ids)
rng.shuffle(unique_sessions)

# Build one row per response so GroupKFold sees the true response-level rows,
# grouped by session_id, matching how B/C/D will actually train.
fold_source = train_features_enriched[["response_id", "session_id"]].copy()
# Re-order by the shuffled session order so fold assignment is seed-dependent.
session_order = {sid: i for i, sid in enumerate(unique_sessions)}
fold_source["_order"] = fold_source["session_id"].map(session_order)
fold_source = fold_source.sort_values("_order").reset_index(drop=True)

gkf = GroupKFold(n_splits=N_FOLDS)
fold_source["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(
    gkf.split(fold_source, groups=fold_source["session_id"])
):
    fold_source.loc[val_idx, "fold"] = fold_idx

assert (fold_source["fold"] >= 0).all(), "every response must get a fold"

cv_folds = (
    fold_source[["session_id", "fold"]]
    .drop_duplicates(subset="session_id")
    .sort_values("session_id")
    .reset_index(drop=True)
)
print(cv_folds["fold"].value_counts().sort_index())


In [ ]:
# --- Verify no session's responses cross fold boundaries ---------------------
check = train_features_enriched.merge(cv_folds, on="session_id", how="left")
folds_per_session = check.groupby("session_id")["fold"].nunique()
assert (folds_per_session == 1).all(), "found sessions whose responses span multiple folds!"
print("OK: every session's responses fall in exactly one fold.")


## 5. Outputs

In [ ]:
session_features.to_csv(OUT_DIR / "session_features.csv", index=False)
train_features_enriched.to_csv(OUT_DIR / "train_features_enriched.csv", index=False)
cv_folds.to_csv(OUT_DIR / "cv_folds.csv", index=False)

for f in ["session_features.csv", "train_features_enriched.csv", "cv_folds.csv"]:
    p = OUT_DIR / f
    print(f"{f}: {p.stat().st_size / 1e6:.2f} MB")


### Feature summary (`feat_*` columns produced)

| Column | What it captures |
|---|---|
| `feat_n_student_words` | Total student word count for the session — raw engagement/verbosity signal. |
| `feat_n_tutor_words` | Total tutor word count for the session. |
| `feat_student_word_share` | Share of session words spoken by the student — who is "doing the talking". |
| `feat_numeric_turns_per_word` | Density of turns containing digits, normalized by session length — proxy for how numeric the session is. |
| `feat_digit_chars_per_word` | Density of digit characters, normalized by session length — reference-notebook feature, complements the turn-level version above. |
| `feat_number_word_count` | Count of spelled-out numbers ("twenty-four") in student turns — captures numeric reasoning not visible in digit form. |
| `feat_hint_count` | Count of tutor turns matching hint/prompting language — proxy for how much scaffolding the tutor provided. |
| `feat_correction_count` | Count of tutor turns matching correction/error language — proxy for how many mistakes the student made along the way. |
| `feat_question_count_tutor` | Number of tutor turns containing a question mark — Socratic-questioning intensity. |
| `feat_question_count_student` | Number of student turns containing a question mark — student confusion/clarification-seeking. |
| `feat_turn_count` | Total tutor+student turns — overall session length. |
| `feat_avg_turn_length_student` | Mean words per student turn — depth of student responses. |
| `feat_avg_turn_length_tutor` | Mean words per tutor turn — depth of tutor explanations. |
| `feat_turn_gap_mean_s` | Mean time gap between consecutive turns — pacing/latency of the conversation. |
| `feat_turn_gap_std_s` | Std. dev. of turn gaps — how uneven the pacing was (long pauses vs. quick back-and-forth). |
| `feat_late_session_student_share` | Student word share in the final third of the session — proxy for whether the student "took over" reasoning near the end. |
| `feat_embed_sim_objective_mean` | Mean cosine similarity (MiniLM-L6-v2 embeddings) between student turns and the session's learning objective(s) — on-topic-ness of the student's contributions. |

All columns are computed at the `session_id` level and repeat across
`response_id` rows sharing a session, per the shared naming convention in
`Context.md`.
